# Good vs. bad support — trust diagnostics for offline policy evaluation

A single offline value estimate (`V_hat`) is dangerous without a **trust contract**.
This notebook shows the *same* routing problem under two logging regimes:

1. **Good support** — an exploratory (epsilon-greedy) baseline policy. The
   candidate policy is well supported by the logs → `support_health == "ok"`.
2. **Bad support** — a near-deterministic baseline policy. There is little
   counterfactual signal → the diagnostics fire and `support_health` becomes
   `high_risk`.

The lesson: **always read `support_health` before `V_hat`.**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

import skdr_eval

RNG_SEED = 1
N = 5000

## A shared problem

Three operators; the observed `service_time` depends on client features. The
**only** thing we vary between scenarios is `explore` — how often the logging
policy deviated from its greedy choice.

In [ ]:
def make_logs(n: int, seed: int, explore: float) -> pd.DataFrame:
    """Logged routing decisions under an epsilon-greedy baseline policy."""
    rng = np.random.RandomState(seed)
    ts = pd.Timestamp("2024-01-01") + pd.to_timedelta(np.arange(n) * 5, unit="min")
    urgency = rng.normal(0.0, 1.0, n)
    tenure = rng.uniform(0.0, 1.0, n)
    ops = ["agent_alpha", "agent_bravo", "agent_charlie"]
    k = len(ops)
    mu = np.stack([
        10.0 + 2.0 * urgency + 3.0 * tenure,
        11.0 - 1.0 * urgency + 1.0 * tenure,
        9.0 + 1.0 * urgency + 4.0 * tenure,
    ], axis=1)
    greedy = mu.argmin(axis=1)
    behavior = np.full((n, k), explore / k)
    behavior[np.arange(n), greedy] += 1.0 - explore
    actions = np.array([rng.choice(k, p=behavior[i]) for i in range(n)])
    service_time = np.maximum(mu[np.arange(n), actions] + rng.normal(0, 1, n), 0.1)
    data = {"arrival_ts": ts, "cli_urgency": urgency, "cli_tenure": tenure}
    for op in ops:
        data[f"{op}_elig"] = np.ones(n, dtype=bool)
    data["action"] = [ops[a] for a in actions]
    data["service_time"] = service_time
    return pd.DataFrame(data)


def evaluate(logs: pd.DataFrame):
    models = {
        "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=1),
        "HistGB": HistGradientBoostingRegressor(max_iter=100, max_depth=6, random_state=1),
    }
    return skdr_eval.evaluate_sklearn_models(
        logs=logs, models=models, n_splits=3, outcome_estimator="hgb",
        random_state=1, policy_train="pre_split", policy_train_frac=0.8,
    )


COLS = ["model", "estimator", "V_hat", "ESS", "pareto_k", "min_pscore", "support_health"]

## 1. Good support — an exploratory baseline (`explore = 0.65`)

In [ ]:
good = evaluate(make_logs(N, RNG_SEED, explore=0.65))
print(good.report[COLS].round(3).to_string(index=False))
print("\nwarnings:")
print(good.warnings[["model", "estimator", "support_health", "warning_codes"]].to_string(index=False))

Every row is `ok`, the importance-weight tail (`pareto_k`) is small, `ESS`
is a large fraction of the sample, and the two candidate models produce
**distinct** `V_hat` values. This is an evaluation you can act on.

## 2. Bad support — a near-deterministic baseline (`explore = 0.05`)

In [ ]:
bad = evaluate(make_logs(N, RNG_SEED, explore=0.05))
print(bad.report[COLS].round(3).to_string(index=False))
print("\nwarnings:")
print(bad.warnings[["model", "estimator", "support_health", "warning_codes"]].to_string(index=False))

Now `support_health` is `high_risk`: the logging policy almost never
explored, so the importance weights have a heavy tail (`pareto_k` large), `ESS`
collapses, and `POOR_OVERLAP` / `HIGH_PARETO_K` / `EXTREME_CLIP` fire. The point
estimate is **not** deployment evidence — even though the estimator still
returns a number.

## 3. The decision rule

In [ ]:
good_health = set(good.report["support_health"])
bad_health = set(bad.report["support_health"])
print("good support_health:", good_health)
print("bad  support_health:", bad_health)

# The contrast is the whole point of this tutorial:
assert good_health == {"ok"}, good_health
assert "high_risk" in bad_health, bad_health
print("\nGood support -> proceed to A/B-test planning.")
print("Bad support  -> improve logging/exploration before trusting the estimate.")

**Takeaway.** `skdr-eval` is designed to make the untrustworthy case
*loud*. When you copy this workflow onto your own logs, read the trust contract
first: `support_health`, then `warnings`, then `sensitivity` and calibration,
and only then `V_hat`. See the
[logs → experiment-review card recipe](https://github.com/dgenio/skdr-eval/blob/main/docs/recipes/logs-to-experiment-card.md)
for the full practitioner workflow.